# Getting started with MemsArrayDB objects

The `MemsArrayDB` class allows getting signals from MemsArray saved in a remote database 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.base import MemsArrayDB

log.setLevel( "INFO" )

# Set database access credentials
DBHOST = 'http://dbwelfare.biimea.io/'
LOGIN = 'ailab'
EMAIL = 'bruno.gas@biimea.com'
PASSWORD = '#T;uZnQ5UJ_JC~&'

## Starting with labelized signals

The ``MemsArrayDB`` constructor connects to the database and populates the antenna parameters with metadata received from the database. An exception is raised if connection failed. The flag ``preload=False`` ensures that the signal is not loaded.

In [2]:
# choose label, file and sequence in file:
LABEL_ID = 18
FILE_ID = 8692          # 5838 (1), 7135 (1), 6860(3), 6560(1)
SEQUENCE_ID = 0  

# Define the antenna
antenna = MemsArrayDB( 
    dbhost=DBHOST, login=LOGIN, email=EMAIL, passwd=PASSWORD, 
    label_id=LABEL_ID, file_id=FILE_ID, sequence_id=SEQUENCE_ID,
    preload=True )



2023-09-18 11:44:32,039 [INFO]:  .Try connecting on endpoint database http://dbwelfare.biimea.io/...
2023-09-18 11:44:32,464 [INFO]:  .Got HTTP 200 status code from server
2023-09-18 11:44:32,465 [INFO]:  .Received CSRF token: c2mjU7iZGLPom9lJma5OCx04G2NFb5vq. Update session with
2023-09-18 11:44:32,466 [INFO]:  .Received session id: mvo145v9hkx2r96dxdjuvqp02mbkm0g2
2023-09-18 11:44:32,466 [INFO]:  .Successfully connected on http://dbwelfare.biimea.io/
2023-09-18 11:44:32,467 [INFO]:  .Downloading metadata for object 'sourcefile' [8692]...
2023-09-18 11:44:32,534 [INFO]:  .Object sourcefile found with identifier [8692] 
2023-09-18 11:44:32,535 [INFO]:  .Set 32 MEMs numbered from 0 to 31
2023-09-18 11:44:32,535 [INFO]:  .Set 0 analog channels numbered from 0 to -1
2023-09-18 11:44:32,536 [INFO]:  .Downloading...
2023-09-18 11:44:32,537 [INFO]:  .Downloading labelized audio files from http://dbwelfare.biimea.io/...
2023-09-18 11:44:32,564 [INFO]:  .Found 4 labelized audio files
2023-09-1

request=/filelabeling/?label=18&limit=100&sourcefile=8692


2023-09-18 11:44:33,068 [INFO]:  .Downloading metadata for object 'sourcefile' [8692]...
2023-09-18 11:44:33,138 [INFO]:  .Object sourcefile found with identifier [8692] 


I'm a NDarray signal with frame size = 80860 and frame number = 1


2023-09-18 11:44:33,397 [INFO]:  .Downloading metadata for object 'sourcefile' [8692]...
2023-09-18 11:44:33,496 [INFO]:  .Object sourcefile found with identifier [8692] 
2023-09-18 11:44:33,558 [INFO]:  .Downloading metadata for object 'sourcefile' [8692]...


I'm a NDarray signal with frame size = 54536 and frame number = 1
I'm a NDarray signal with frame size = 37671 and frame number = 1


2023-09-18 11:44:33,638 [INFO]:  .Object sourcefile found with identifier [8692] 
2023-09-18 11:44:33,831 [INFO]:  .Trying to disconnect from database http://dbwelfare.biimea.io/...
2023-09-18 11:44:33,879 [INFO]:  .Logout successful.
2023-09-18 11:44:33,880 [INFO]: Got 80860 samples on 2 MEMs


I'm a NDarray signal with frame size = 67293 and frame number = 1


In [14]:
print( 'sf = ', antenna.sampling_frequency )
print( 'available mems = ', antenna.available_mems )
print( 'counter: ', antenna.counter )
print( 'counter skipping: ', antenna.counter_skip )
antenna_definition = np.load ('Antenna-square-JetsonNano-0001.npy', allow_pickle=True )
antenna.setMemsPosition( antenna_definition.item().get('positions') )
antenna.setActiveMems( antenna_definition.item().get('mems'))
print( 'active mems = ', antenna.mems )


2023-09-18 11:54:21,064 [WARNING]: in megamicros.log (base.py:259): Positions array of size (25, 3) does not match with the activated MEMs list: deactivating MEMs
2023-09-18 11:54:21,065 [INFO]:  .Set a 25 activable MEMs antenna with physical positions
2023-09-18 11:54:21,067 [INFO]:  .25 MEMs were activated among 0 to 31 available MEMs


sf =  10000.0
available mems =  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
counter:  True
counter skipping:  False
active mems =  [0, 1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 22, 23, 24, 25, 28, 29, 30, 31]


## With initialization parameters 

In [ ]:
# Create a 10 MEMs antenna
antenna = MemsArray( available_mems_number=10, mems=[0, 1, 2] )

## With antenna physical parameters

In [ ]:
# Create an antenna from physical parameters
antenna_definition = np.load ('Antenna-square-JetsonNano-0001.npy', allow_pickle=True )
mems_position = antenna_definition.item().get('positions')
antenna = MemsArray( mems_position=mems_position )

# Create antenna and set activated MEMs 
antenna = MemsArray( 
    available_mems_number = len( antenna_definition.item().get('available_mems') ),
    mems_position=mems_position,
    mems=antenna_definition.item().get('mems') 
)

## Plot the antenna MEMs

In [ ]:
mems_position = antenna.mems_position
fig = plt.figure()
ax = fig.add_subplot( 121, projection='3d' )
ax.scatter( mems_position[:,0], mems_position[:,1], mems_position[:,2], marker='^' )

## Generate some signals

Since no input stream is declared, the antenna generates some random samples from a uniform distribution over [-1, +1].
No frame size has been specified either. Default is given by ``megamicros.base.DEFAULT_FRAME_LENGTH`` which is set to 256.

In [ ]:
# iterate over the antenna data stream
for i, data in enumerate( antenna ):
    print( f"data({data.shape})={data}")
    if i > 10:
        break

## Setting a DB input stream

Setting a DB stream as input consists in connecting the Antenna to a Megamicros remote database with the adequat address and request.
In what follows, one ask the data base to send the first sequence from the file number 8692 for which label is 18:

In [ ]:
antenna.setInputDB( label_id = 18, file_id=8692, sequence_id=0 )

## Getting signal from DB

In [ ]:
# Available labels
LABEL_SOW_FEEDING_CALL = 18
LABEL_PIGLET_SQUEALS = 15
LABEL_SOW_GRUNT_NERVOUS = 16
LABEL_ROOM_NOISE = 29
LABEL_SOW_GRUNT = 8
LABEL_SOW_GRUNT_MODSTRESS  = 1
LABEL_SOW_SCREAMS = 3
LABEL_PIGLET_SQUEALS_2 = 5

# choose label, file and sequence in file:
LABEL_ID = LABEL_SOW_FEEDING_CALL
FILE_ID = 8692          # 5838 (1), 7135 (1), 6860(3), 6560(1)
SEQUENCE_ID = 0         

with AidbSession(
    dbhost='http://dbwelfare.biimea.io/',
    login='ailab',
    email='bruno.gas@biimea.com',
    password='#T;uZnQ5UJ_JC~&' ) as session:
        signal: MuAudio = session.load_labelized( 
            sourcefile_id=FILE_ID, 
            label_id=LABEL_ID, 
            limit=100, 
            channels=list( np.arange( 32 ) + 1 ) 
        )[SEQUENCE_ID]

# get infos
LABEL_TXT = signal.label
CHANNELS_NUMBER = signal.channels_number
SAMPLES_NUMBER = signal.samples_number
SAMPLING_FREQUENCY = signal.sampling_frequency

print( f"Some informations about the signal loaded:" )
print( f" > label={LABEL_TXT}" )
print( f" > channels_number={CHANNELS_NUMBER}" )
print( f" > samples_number={SAMPLES_NUMBER}" )
print( f" > sampling_frequency={SAMPLING_FREQUENCY}" )

In [ ]:
# Play sound using channel 0 and 1
left = np.array( signal.channel(0) )
right = np.array( signal.channel(1) )
sound = np.array( [left, right] ).T

display.Audio( sound, rate=SAMPLING_FREQUENCY )

## Set the beamformer

In [ ]:
FRAME_LENGTH = 1024
AREA = [12, 14, 0.01]
AREA_QUANTIZATION = [4, 4, 1/0.01]

# Get the antenna physical description
antenna = np.load ('Antenna-square-JetsonNano-0001.npy', allow_pickle=True )
mems_position = antenna.item().get("positions")

# Create the beamformer
bmf = Beamformer( 
    mems_position = mems_position,
    sampling_frequency = SAMPLING_FREQUENCY,
    window_size = FRAME_LENGTH,    
    area = AREA,
    area_quantization = AREA_QUANTIZATION
)

# Move the antenna in the right place:
bmf.moveArea( [0, 0, -2] )

# Limit the frequency bandwidth for BF computing
bmf.setBandWidth( [200, 2000], unit="frequency" )

# Init the beamformer
bmf.init()

# print area locations and antenna 
space_locations = bmf.getLocations()
mems_location = bmf.getMems()
nx, ny, nz = bmf.getLocationsNumber()

fig = plt.figure()
ax = fig.add_subplot( 121, projection='3d' )
ax.scatter( space_locations[:,0], space_locations[:,1], space_locations[:,2] )
ax.scatter( mems_location[:,0], mems_location[:,1], mems_location[:,2], marker='^' )
ax = fig.add_subplot( 122, projection='3d' )
ax.scatter( mems_location[:,0], mems_location[:,1], mems_location[:,2], marker='^' )
fig.show()


## Compute preformed channels 

In [ ]:
# Get the whole 32 channels signal as a numpy.ndarray
signal32 = signal().T

# Check if some available mems have not been activated
# Remove from signal if any
mems = antenna.item().get('mems')
available_mems = antenna.item().get('available_mems')
if False in np.isin( available_mems, mems ):
    mask = list( np.invert( np.logical_not( np.isin( available_mems, mems ) ) ) )
    signal32 = signal32[:,mask]

FRAMES_NUMBER = SAMPLES_NUMBER // FRAME_LENGTH - 1

print( f"{FRAMES_NUMBER} frames of {FRAME_LENGTH} samples to perform... " )


imgs = []
for i in range( FRAMES_NUMBER ):
    bf = bmf.beamform( signal32[i*FRAME_LENGTH:(i+1)*FRAME_LENGTH,:] )
    imgs.append( np.reshape( bf, (nx, ny) ) )

### Make video

In [ ]:
generate_moovie( 
    imgs, 
    rate=SAMPLING_FREQUENCY/FRAME_LENGTH,
    sound=sound.astype( np.float32 ).T,
    sampling_frequency=SAMPLING_FREQUENCY,
    norm=None,
    extent=( 0, AREA[0], 0, AREA[1] ),
    cleanup=True
)

### Do the same with an antenna object

In [ ]:
# Declare a MEMs antenna
antenna = MemsArray( available_mems_number=CHANNELS_NUMBER )

# set active mems
antenna.setActiveMems( [i for i in range( CHANNELS_NUMBER )] )
print( f"active mems number={antenna.mems_number}" )

# iterate over the antenna data stream
#for i, data in enumerate( antenna ):
#    print( f"data={data}")
#    if i > 10:
#        break
